# CAR-bench Agent Alignment: Odds Ratio Preference Optimization (ORPO) Pipeline

This notebook contains a complete training pipeline to align a Large Language Model (LLM) for the CAR-bench voice assistant competition using Odds Ratio Preference Optimization (ORPO).

## Pipeline Architecture
ORPO aligns language models by combining supervised fine-tuning (SFT) and preference learning into a single loss function. Unlike Direct Preference Optimization (DPO), ORPO does not require a separate SFT phase or a reference model. This significantly reduces GPU memory (VRAM) overhead and training time, making it suitable for resource-constrained environments like Google Colab and Kaggle.

## Contents
1. Environment Setup: Install packages and clone repository files.
2. Risk Management: Establish connection keep-alive, persistent storage, and memory recovery safeguards.
3. Data Preparation: Load datasets and parse preference pairs (chosen and rejected turns).
4. Formatting and Tokenization: Apply chat templates for prompt-chosen-rejected mapping.
5. Model Loading: Initialize base weights and configure PEFT/LoRA adapters.
6. Training Configuration: Set training hyperparameters and execute ORPO with automatic checkpoint recovery.
7. Visualization: Plot training losses and reward accuracies.
8. Saving and Merging Model Weights: Save adapters and merge to 16-bit precision weights.
9. Model Quantization and GGUF Export: Quantize to GGUF format.
10. Downloading Results and Checkpoints: Zip and download artifacts.
11. Inference Server Deployment: Launch local server for evaluation.

## CAR-bench Leaderboard & Baselines

Below are the baseline results for frontier and open-source models on the CAR-bench voice assistant tasks. The goal of this alignment pipeline is to fine-tune a smaller, local model (e.g. Qwen2.5-3B-Instruct) to match or exceed these baselines by specializing on the 19 vehicle assistant policies:

| Model | Alignment Method | Avg Pass^3 | Base Pass^3 | Hall. Pass^3 | Disamb. Pass^3 | Source/Platform |
|---|---|---|---|---|---|---|
| Claude Opus 4.6 | RLHF (Frontier) | 0.58 | 0.80 | 0.48 | 0.46 | Anthropic (Official) |
| GPT-5 | RLHF (Frontier) | 0.54 | 0.66 | 0.60 | 0.36 | OpenAI (Official) |
| Gemini 2.5 Pro | RLHF (Frontier) | 0.38 | 0.53 | 0.34 | 0.28 | Google (Official) |
| **Qwen2.5-3B-Instruct (Fine-Tuned)** | **ORPO (This Run)** | **TBD** | **TBD** | **TBD** | **TBD** | **Local / Cloud Run** |
| Qwen3-32B | Base SFT/RLHF | 0.31 | 0.45 | 0.27 | 0.22 | Alibaba (Official) |
| xLAM-2-32B | SFT / DPO | 0.16 | 0.26 | 0.11 | 0.12 | Salesforce (Official) |

## 1. Environment Setup

In [ ]:
# RESTRICTED ORDER OF IMPORTS - Unsloth must be imported BEFORE transformers/peft/trl
try:
    import unsloth
except ImportError:
    pass

# Check if running in Google Colab or Kaggle
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Clone repository if running in a clean cloud VM
import os
if IN_COLAB or not os.path.exists("src"):
    print("Cloning repository to retrieve wiki.md policy and scripts...")
    !git clone --recursive https://github.com/CAR-bench/car-bench-ijcai.git
    %cd car-bench-ijcai
else:
    print("Running in local workspace.")

# Robust check/fix for submodule/dependency content in third_party/car-bench
if not os.path.exists("third_party/car-bench/pyproject.toml"):
    print("Submodule files missing in third_party/car-bench. Attempting to restore...")
    !git submodule update --init --recursive
    if not os.path.exists("third_party/car-bench/pyproject.toml"):
        print("Directly cloning official car-bench dependency...")
        import shutil
        if os.path.exists("third_party/car-bench"):
            try:
                shutil.rmtree("third_party/car-bench")
            except Exception:
                !rm -rf third_party/car-bench
        !git clone --depth 1 https://github.com/CAR-bench/car-bench.git third_party/car-bench


In [ ]:
# Install dependency packages
!pip install -q transformers peft trl datasets accelerate bitsandbytes huggingface_hub protobuf openai python-dotenv matplotlib seaborn

# Install Unsloth for optimized training speed
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.27"

In [ ]:
# Restructure imports: Unsloth must be imported before any transformers/trl/peft libraries
try:
    from unsloth import FastLanguageModel
except ImportError:
    pass

import os
import sys
import json
import torch
import random

# Establish seeds for training reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Load OpenAI API key from environment
if "OPENAI_API_KEY" not in os.environ:
    print("OPENAI_API_KEY not found in environment. Please provide it or press enter to skip:")
    api_key = getpass.getpass("OPENAI_API_KEY: ")
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key

print("API Key configured:", "Yes" if os.environ.get("OPENAI_API_KEY") else "No")

## 2. Risk Management & Persistent Storage

To prevent training failures due to timeouts, RAM overflows, or cloud VM disconnects, we establish the following safeguards:
1. Keep-Alive Script: Javascript to prevent notebook inactivity disconnects.
2. Google Drive Mounting: Saves checkpoints directly to persistent storage (Google Drive or Kaggle outputs) rather than transient VM storage.
3. Memory Cleanup: Garbage collection and GPU cache purge to avoid Out-Of-Memory (OOM) errors.

### Keep-Alive Script
Paste the following JavaScript code into the browser developer console (F12) to simulate clicks and prevent timeouts during long training sessions:

```javascript
function KeepAlive() {
    console.log("Simulating interaction to keep session active...");
    var connectButton = document.querySelector("colab-connect-button");
    if (connectButton) connectButton.click();
}
setInterval(KeepAlive, 60000);
```

In [ ]:
# Mount Google Drive if running in Colab to preserve checkpoint directories
PERSISTENT_DIR = "./outputs"
if IN_COLAB:
    try:
        from google.colab import drive
        print("Mounting Google Drive for persistent checkpoints...")
        drive.mount('/content/drive')
        PERSISTENT_DIR = "/content/drive/MyDrive/car_bench_checkpoints"
    except Exception as e:
        print(f"Google Drive mount skipped or failed: {e}. Checkpoints will be saved locally.")
elif os.path.exists("/kaggle/working"):
    PERSISTENT_DIR = "/kaggle/working/checkpoints"
    
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target checkpoints directory configured: {PERSISTENT_DIR}")

In [ ]:
import gc

def clear_gpu_memory():
    """Purges system garbage collection and releases cached PyTorch CUDA memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Released GPU cache memory.")

clear_gpu_memory()

## 3. Dataset Download & Exploration

We download the official CAR-bench dataset from Hugging Face at [johanneskirmayr/car-bench-dataset](https://huggingface.co/datasets/johanneskirmayr/car-bench-dataset) to cache the training and validation splits locally before preprocessing.


In [ ]:
from datasets import load_dataset
import os

print("Downloading CAR-bench dataset from Hugging Face (johanneskirmayr/car-bench-dataset)...")
configs = ["tasks_base", "tasks_disambiguation", "tasks_hallucination"]
splits = ["train", "test"]

os.makedirs("data", exist_ok=True)

for config in configs:
    for split in splits:
        print(f"Downloading split: config={config}, split={split}...")
        ds = load_dataset("johanneskirmayr/car-bench-dataset", config, split=split)
        print(f"  Successfully loaded {len(ds)} samples.")
        
        # Save raw dataset copy locally for verification and exploration
        raw_path = f"data/raw_{config}_{split}.jsonl"
        ds.to_json(raw_path)
        print(f"  Saved locally to: {raw_path}")


## 4. Data Preparation

We load training data and construct preference pairs containing a `prompt` (conversation history), `chosen` (correct turn response), and `rejected` (suboptimal or incorrect turn response).

In [ ]:
import os
import json
from datasets import Dataset, load_dataset

# Configuration paths
local_mapped_path = "data/dpo_dataset_hf_mapped.jsonl"
os.makedirs("data", exist_ok=True)

dataset = None

# Step 1: Check if already preprocessed and cached on disk to prevent loss on OOM/crash
if os.path.exists(local_mapped_path):
    print(f"Loading cached preprocessed dataset from {local_mapped_path}...")
    try:
        dataset = load_dataset("json", data_files=local_mapped_path, split="train")
    except Exception as e:
        print(f"Error loading cached dataset: {e}")

# Step 2: Fallback to local logs parse if available
if not dataset or len(dataset) == 0:
    dpo_dataset_path = "data/dpo_dataset.jsonl"
    if os.path.exists(dpo_dataset_path):
        print(f"Loading local preference dataset from {dpo_dataset_path}...")
        try:
            dataset = load_dataset("json", data_files=dpo_dataset_path, split="train")
        except Exception as e:
            print(f"Error loading local json dataset: {e}")

# Step 3: Fallback to Hugging Face dataset download + preprocessing mapping + caching
if not dataset or len(dataset) == 0:
    print("Preprocessed cached dataset not found. Downloading and preprocessing from Hugging Face johanneskirmayr/car-bench-dataset...")
    try:
        from datasets import load_dataset as hf_load_dataset
        import random
        from openai import OpenAI
        
        splits = ["tasks_base", "tasks_disambiguation", "tasks_hallucination"]
        dpo_samples = []
        
        # Load environment API Key
        api_key = os.environ.get("OPENAI_API_KEY")
        client = OpenAI(api_key=api_key) if api_key else None
        if not client:
            print("OPENAI_API_KEY not set. Falling back to heuristic mutations for negative examples.")
            
        wiki_file = "third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
        if not os.path.exists(wiki_file):
            wiki_file = "car-bench-ijcai/third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
            if not os.path.exists(wiki_file):
                wiki_file = "car-bench-ijcai-vsf/third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
        
        wiki_raw = ""
        if os.path.exists(wiki_file):
            with open(wiki_file, "r", encoding="utf-8") as f:
                wiki_raw = f.read()
                
        for config_name in splits:
            for split_name in ["train", "test"]:
                try:
                    print(f"Loading and mapping Hugging Face split: {config_name} ({split_name})...")
                    ds = hf_load_dataset("johanneskirmayr/car-bench-dataset", config_name, split=split_name)
                    for item in ds:
                        instruction = item.get("instruction", "")
                        context_init = item.get("context_init_config", "{}")
                        if isinstance(context_init, str):
                            try:
                                context_init_data = json.loads(context_init)
                            except Exception:
                                context_init_data = {}
                        else:
                            context_init_data = context_init
                            
                        # Build system prompt with context placeholders replaced
                        if wiki_raw:
                            location_str = json.dumps(context_init_data.get("current_location", {}), separators=(',', ':'))
                            datetime_str = json.dumps(context_init_data.get("current_datetime", {}), separators=(',', ':'))
                            system_prompt = wiki_raw.replace("INS:", "").replace("AUT-POL:", "").replace("LLM-POL:", "")
                            system_prompt = system_prompt.replace("{{placeholder_location_based_on_task_context_init_config}}", location_str)
                            system_prompt = system_prompt.replace("{{placeholder_datetime_based_on_task_context_init_config}}", datetime_str)
                        else:
                            system_prompt = "You are a helpful in-car voice assistant."
                            
                        actions_str = item.get("actions", "[]")
                        chosen_content = f"Thought: I need to execute the following actions.\n{actions_str}"
                        
                        prompt_history = [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": instruction}
                        ]
                        
                        # Generate rejected content
                        rejected_content = None
                        if client:
                            rule = random.choice([
                                "Call a tool without getting a prior confirmation from the user (violates user confirmation rule).",
                                "Call a planning_tool directly (violates forbidden tool rule).",
                                "Output reasoning thoughts in plaintext rather than JSON (violates output format rule).",
                                "Return incorrect coordinates or search fields (violates accuracy rule)."
                            ])
                            try:
                                chat_completion = client.chat.completions.create(
                                    model="gpt-4o-mini",
                                    messages=[
                                        {"role": "system", "content": "You are a database alignment assistant. Generate a negative example violating the specified rule."},
                                        {"role": "user", "content": f"Take the following successful assistant response and rewrite it to violate the rule: {rule}\n\nResponse:\n{chosen_content}"}
                                    ],
                                    temperature=0.7,
                                    max_tokens=800
                                )
                                rejected_content = chat_completion.choices[0].message.content.strip()
                            except Exception as api_err:
                                print(f"API generation error: {api_err}. Falling back to heuristic mutation.")
                                
                        if not rejected_content:
                            try:
                                actions = json.loads(actions_str)
                                if actions:
                                    actions[0]["name"] = "planning_tool"
                                rejected_content = f"Thought: I need to execute the following actions.\n{json.dumps(actions)}"
                            except Exception:
                                rejected_content = f"Thought: I need to execute the following actions.\n[]"
                                
                        dpo_samples.append({
                            "prompt": prompt_history,
                            "chosen": [{"role": "assistant", "content": chosen_content}],
                            "rejected": [{"role": "assistant", "content": rejected_content}]
                        })
                except Exception as e:
                    print(f"Skipped HF config {config_name} {split_name}: {e}")
                    
        dataset = Dataset.from_list(dpo_samples)
        
        # Cache the preprocessed dataset immediately to disk to prevent loss on OOM
        print(f"Caching preprocessed preference dataset to {local_mapped_path}...")
        dataset.to_json(local_mapped_path)
    except Exception as e:
        print(f"Failed to load dataset from Hugging Face: {e}")
        dataset = Dataset.from_list([])

print(f"Preference dataset loaded. Total samples: {len(dataset)}")


## 5. Formatting and Tokenization

We apply the tokenizer chat template to format the prompt history and responses into a single string for training.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_orpo_data(examples):
    formatted_prompts = []
    formatted_chosen = []
    formatted_rejected = []
    
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        # Format prompt history (ends with assistant start token)
        p_text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        
        # Format assistant turns
        c_text = tokenizer.apply_chat_template(chosen, tokenize=False, add_generation_prompt=False)
        r_text = tokenizer.apply_chat_template(rejected, tokenize=False, add_generation_prompt=False)
        
        # Remove prompt prefix if tokenizer duplicates it
        if c_text.startswith(p_text):
            c_text = c_text[len(p_text):]
        if r_text.startswith(p_text):
            r_text = r_text[len(p_text):]
            
        formatted_prompts.append(p_text)
        formatted_chosen.append(c_text)
        formatted_rejected.append(r_text)
        
    return {
        "prompt": formatted_prompts,
        "chosen": formatted_chosen,
        "rejected": formatted_rejected
    }

if len(dataset) > 0:
    formatted_dataset = dataset.map(format_orpo_data, batched=True)
    print("Example Formatted DPO Sample:")
    print(f"Prompt:\n{formatted_dataset[0]['prompt'][:300]}...\n")
    print(f"Chosen:\n{formatted_dataset[0]['chosen']}\n")
    print(f"Rejected:\n{formatted_dataset[0]['rejected']}")
else:
    formatted_dataset = dataset
    print("No data available to format.")

## 6. Model Loading and PEFT Configuration


In [ ]:
# Pre-loading clean memory check
clear_gpu_memory()

USE_UNSLOTH = True
MAX_SEQ_LENGTH = 2048

if USE_UNSLOTH:
    print("Loading model via Unsloth...")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    print("Loading model via Standard HF PEFT...")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

print("Model initialized. Trainable parameters:")
model.print_trainable_parameters()

## 7. Training Configuration and ORPO Execution

We define the training arguments using the persistent storage directory configured in the Risk Management section. We also check for existing checkpoints to automatically resume execution if a server restart occurs.

In [ ]:
from trl import ORPOTrainer, ORPOConfig

# Check for existing checkpoints to resume training automatically
resume_checkpoint = None
if os.path.exists(PERSISTENT_DIR):
    checkpoints = [os.path.join(PERSISTENT_DIR, d) for d in os.listdir(PERSISTENT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        # Sort checkpoints by step iteration count
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_checkpoint = checkpoints[-1]
        print(f"Found existing checkpoint: {resume_checkpoint}. Resuming from this step.")

orpo_config = ORPOConfig(
    output_dir=PERSISTENT_DIR,
    learning_rate=5e-6,      # Recommended range: 5e-6 to 1e-5
    beta=0.1,                # Odds Ratio loss multiplier (lambda)
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    save_strategy="epoch",    # Save checkpoints after each epoch
    save_total_limit=2,      # Retain only the last 2 checkpoints to manage disk bounds
    logging_steps=5,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    report_to="none",
    average_tokens_across_devices=False
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=formatted_dataset,
    tokenizer=tokenizer
)

if len(formatted_dataset) > 0:
    print("Starting ORPO training...")
    try:
        trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
        print("Training completed successfully!")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("\nCUDA Out Of Memory encountered. Suggested mitigation steps:")
            print("1. Set MAX_SEQ_LENGTH to 1024 or 512.")
            print("2. Set per_device_train_batch_size to 1.")
            print("3. Execute 'clear_gpu_memory()' and restart execution cell.")
        raise e
else:
    print("Dataset is empty. Add preference samples to dataset before running.")

## 8. Training Metrics Visualization

We plot SFT loss, Odds Ratio loss, and accuracy metrics collected during training iterations.

In [ ]:
import matplotlib.pyplot as plt

if hasattr(trainer, "state") and trainer.state.log_history:
    history = trainer.state.log_history
    
    steps = []
    sft_losses = []
    or_losses = []
    total_losses = []
    accuracies = []
    
    for log in history:
        if "step" in log:
            if "loss" in log:
                steps.append(log["step"])
                total_losses.append(log["loss"])
            if "sft_loss" in log:
                sft_losses.append(log["sft_loss"])
            if "odds_ratio_loss" in log:
                or_losses.append(log["odds_ratio_loss"])
            if "rewards/accuracies" in log:
                accuracies.append(log["rewards/accuracies"])
                
    plt.figure(figsize=(15, 5))
    
    # Loss Curves Plot
    plt.subplot(1, 2, 1)
    if total_losses:
        plt.plot(steps[:len(total_losses)], total_losses, label="Total Loss", color="red")
    if sft_losses:
        plt.plot(steps[:len(sft_losses)], sft_losses, label="SFT Loss", color="blue")
    if or_losses:
        plt.plot(steps[:len(or_losses)], or_losses, label="Odds Ratio Loss", color="purple")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Loss Curves")
    plt.legend()
    plt.grid(True)
    
    # Accuracy Curve Plot
    plt.subplot(1, 2, 2)
    if accuracies:
        plt.plot(steps[:len(accuracies)], accuracies, label="Accuracy", color="green")
        plt.xlabel("Step")
        plt.ylabel("Accuracy")
        plt.title("Preference Accuracy")
        plt.legend()
        plt.grid(True)
        
    plt.tight_layout()
    plt.show()
else:
    print("No log history found to plot.")

## 9. Saving and Merging Model Weights

We save the LoRA adapter weights and merge them into the base model weights to produce a 16-bit precision checkpoint for deployment.

In [ ]:
ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "lora_adapter")
MERGED_PATH = os.path.join(PERSISTENT_DIR, "merged_model")

# Save training adapter checkpoints
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter weights saved to {ADAPTER_PATH}")

# Save merged weights (16-bit precision)
if USE_UNSLOTH:
    print("Saving merged 16-bit model weights via Unsloth...")
    model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to {MERGED_PATH}")
else:
    print("Merging standard HF PEFT model weights...")
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16,
        device_map="cpu",
        trust_remote_code=True
    )
    peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    merged_model = peft_model.merge_and_unload()
    
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    print(f"Merged model saved to {MERGED_PATH}")

## 10. Model Quantization and GGUF Export

When using Unsloth, weights can be exported directly to GGUF format for memory-efficient local deployment.

In [ ]:
if USE_UNSLOTH:
    GGUF_PATH = os.path.join(PERSISTENT_DIR, "merged_q4_k_m")
    print("Exporting model to Q4_K_M GGUF format...")
    try:
        model.save_pretrained_merged(GGUF_PATH, tokenizer, save_method="quantized_gguf", quantization_method="q4_k_m")
        print(f"GGUF model exported to {GGUF_PATH}-unsloth.Q4_K_M.gguf")
    except Exception as e:
        print(f"Export to GGUF failed: {e}")
        
    # Provide download option in Colab environments
    if IN_COLAB:
        try:
            from google.colab import files
            files.download(f"{GGUF_PATH}-unsloth.Q4_K_M.gguf")
        except Exception as e:
            print(f"Download trigger failed: {e}")
else:
    print("GGUF export is only supported when running with Unsloth.")

## 11. Downloading Results and Checkpoints

This section provides helper functions to compress (zip) and download model checkpoints, adapter files, or full training outputs. This is especially useful in Google Colab (triggers browser download) and Kaggle (saves zip file to outputs for easy download via sidebar).

In [ ]:
import shutil
import os

def zip_and_download(folder_path, output_filename):
    """Zips a folder and triggers download in Colab, or creates a downloadable file in Kaggle/local."""
    if not os.path.exists(folder_path):
        print(f"Folder '{folder_path}' does not exist. Skipping compression.")
        return
        
    zip_path = f"{output_filename}.zip"
    print(f"Zipping '{folder_path}' into '{zip_path}'...")
    
    # Zip the folder
    shutil.make_archive(output_filename, 'zip', folder_path)
    file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Created: {zip_path} ({file_size_mb:.2f} MB)")
    
    if IN_COLAB:
        try:
            from google.colab import files
            print("Initiating browser download of the zip file...")
            files.download(zip_path)
        except Exception as e:
            print(f"Automatic download failed: {e}. Download manually from the Colab file explorer.")
    else:
        print(f"Zip file is ready for download/copy at: {os.path.abspath(zip_path)}")

# Choose what you want to download:
# Option 1: Download LoRA adapter only (recommended, small size)
zip_and_download(ADAPTER_PATH, "lora_adapter_results")

# Option 2: Download full training outputs/checkpoints folder (large size)
# zip_and_download(PERSISTENT_DIR, "all_training_checkpoints")

## 12. Inference Server Deployment

Serve the model locally using vLLM to integrate with the evaluation harness:

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model ./merged_model \
    --port 8000 \
    --dtype float16 \
    --gpu-memory-utilization 0.90
```

Configure environmental variables in `.env`:
```env
AGENT_LLM=openai/car-bench-llm
OPENAI_API_BASE=http://localhost:8000/v1
OPENAI_API_KEY=local-dummy-key
```

Execute the evaluation harness:
```bash
uv run car-bench-run --scenario scenarios/track_1_agent_under_test/local_smoke.toml
```